In [3]:
import os

data_dir = "../data/raw/Task01_BrainTumour"

print(os.listdir(data_dir))

['Task01_BrainTumour.tar', 'labelsTr', 'imagesTr', 'imagesTs', 'dataset.json']


In [4]:
import os

images_dir = os.path.join(data_dir, "imagesTr")
labels_dir = os.path.join(data_dir, "labelsTr")

image_files = sorted(os.listdir(images_dir))
label_files = sorted(os.listdir(labels_dir))

print(image_files[0])
print(label_files[0])

BRATS_001.nii.gz
BRATS_001.nii.gz


確認影像和標籤是否對應

In [5]:
images_dir = "../data/raw/Task01_BrainTumour/imagesTr"
labels_dir = "../data/raw/Task01_BrainTumour/labelsTr"

image_files = sorted(os.listdir(images_dir))
label_files = sorted(os.listdir(labels_dir))


data_dicts = [
    {
        "image": os.path.join(images_dir, img),
        "label": os.path.join(labels_dir, lbl)
    }
    for img, lbl in zip(image_files, label_files)
]


data_dicts[:2]

[{'image': '../data/raw/Task01_BrainTumour/imagesTr/BRATS_001.nii.gz',
  'label': '../data/raw/Task01_BrainTumour/labelsTr/BRATS_001.nii.gz'},
 {'image': '../data/raw/Task01_BrainTumour/imagesTr/BRATS_002.nii.gz',
  'label': '../data/raw/Task01_BrainTumour/labelsTr/BRATS_002.nii.gz'}]

In [6]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd
)


train_transforms = Compose(
    [
        LoadImaged(keys=["image", "label"]),
        EnsureChannelFirstd(keys=["image", "label"]),
    ]
)

In [7]:
sample = train_transforms(data_dicts[0])

sample.keys()

dict_keys(['image', 'label'])

In [8]:
print(type(sample["image"]))
print(type(sample["label"]))

<class 'monai.data.meta_tensor.MetaTensor'>
<class 'monai.data.meta_tensor.MetaTensor'>


In [9]:
print(sample["image"].shape)
print(sample["label"].shape)

torch.Size([4, 240, 240, 155])
torch.Size([1, 240, 240, 155])


In [10]:
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    NormalizeIntensityd,
)
train_transforms = Compose(
    [
        # 讀取 NIfTI
        LoadImaged(keys=["image", "label"]),

        # Channel 放到第一維
        EnsureChannelFirstd(keys=["image", "label"]),

        # 統一醫學影像方向
        Orientationd(
            keys=["image", "label"],
            axcodes="RAS",
        ),

        # 統一 voxel spacing
        Spacingd(
            keys=["image", "label"],
            pixdim=(1.0, 1.0, 1.0),
            mode=("bilinear", "nearest"),
        ),

        # MRI 強度標準化
        NormalizeIntensityd(
            keys="image",
            nonzero=True,
            channel_wise=True,
        ),
    ]
)

/opt/anaconda3/envs/medical-ai/lib/python3.11/site-packages/monai/utils/deprecate_utils.py:320: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [11]:
sample = train_transforms(data_dicts[0])

In [12]:
print(sample["image"].shape)
print(sample["label"].shape)

torch.Size([4, 240, 240, 155])
torch.Size([1, 240, 240, 155])


In [13]:
print(sample["image"].meta["affine"])
print(sample["image"].meta["spatial_shape"])
print(sample["image"].meta["space"])

tensor([[1., 0., 0., 0.],
        [0., 1., 0., 0.],
        [0., 0., 1., 0.],
        [0., 0., 0., 1.]], dtype=torch.float64)
[240 240 155]
RAS


In [14]:
image = sample["image"]

print(image.min())
print(image.max())
print(image.mean())
print(image.std())

metatensor(-3.7696)
metatensor(6.6538)
metatensor(-1.7282e-08)
metatensor(0.4083)


mean結果接近0>>成功將影像中心化
std沒有接近1
因為是用nonzero=True,是設計只利用非零 voxel 計算平均與標準差
但image.std()會把背景值計算進去做平均和標準差
特別注意：背景不參與計算平均與標準差！！
所以改成以下方式去除背景數值

In [15]:
image = sample["image"]

for c in range(image.shape[0]):
    voxels = image[c][image[c] != 0]

    print(f"Channel {c}")
    print("Mean:", voxels.mean().item())
    print("Std :", voxels.std().item())
    print("-" * 30)

Channel 0
Mean: 5.251871471045888e-08
Std : 1.0000003576278687
------------------------------
Channel 1
Mean: -3.3844537483673776e-07
Std : 1.0000003576278687
------------------------------
Channel 2
Mean: -2.0991045346363535e-08
Std : 1.0000003576278687
------------------------------
Channel 3
Mean: -7.870954732425162e-08
Std : 1.0000003576278687
------------------------------
